# SageMaker Deployment — Predictive Maintenance ML Pipeline

This notebook documents how to migrate the locally trained XGBoost model to AWS SageMaker.
The live system uses a local joblib model with graceful fallback — this notebook shows
the complete SageMaker workflow for production cloud deployment.

## Architecture

    Local Training (train.py)
│
▼
models/best_model.pkl (joblib)
│
├── Local Inference (default, always works)
│       └── src/predictor.py → predict_local()
│
└── SageMaker Inference (when SAGEMAKER_ENDPOINT_NAME is set)
└── src/predictor.py → predict_sagemaker()

## What SageMaker adds
- Managed, scalable inference endpoint
- No server management — AWS handles infrastructure
- Auto-scaling based on traffic
- Model versioning and A/B testing support
- CloudWatch monitoring out of the box

In [2]:
# Step 1: Verify AWS credentials and boto3 connection
import boto3
import sagemaker
import sys
sys.path.append('..')

# Check AWS identity
sts = boto3.client('sts', region_name='us-east-1')
identity = sts.get_caller_identity()
print(f"AWS Account: {identity['Account']}")
print(f"IAM User: {identity['Arn']}")
print(f"boto3 version: {boto3.__version__}")
print("AWS credentials configured correctly.")

AWS Account: 751948408724
IAM User: arn:aws:iam::751948408724:user/satvik-sagemaker-user
boto3 version: 1.42.89
AWS credentials configured correctly.


In [3]:
# Step 2: Show the fallback logic in action
# This demonstrates the graceful fallback pattern
import sys
sys.path.append('..')

from src.predictor import predict

# Test prediction — will use local joblib since no endpoint is configured
test_input = {
    "type": "M",
    "air_temperature": 298.1,
    "process_temperature": 308.6,
    "rotational_speed": 1551.0,
    "torque": 42.8,
    "tool_wear": 0.0
}

result = predict(test_input)
print(f"Prediction result: {result}")
print(f"Inference backend used: {result['inference_backend']}")
print()

# Test with high-risk inputs
high_risk_input = {
    "type": "L",
    "air_temperature": 304.0,
    "process_temperature": 314.0,
    "rotational_speed": 1200.0,
    "torque": 70.0,
    "tool_wear": 250.0
}

result_high = predict(high_risk_input)
print(f"High risk prediction: {result_high}")
print(f"Inference backend used: {result_high['inference_backend']}")

FileNotFoundError: Required artifact not found: 'models/best_model.pkl'. Please run the training pipeline first.

In [4]:
# Step 2: Show the fallback logic in action
import sys
import os
sys.path.append('..')
os.chdir('..')  # Move to project root so model paths resolve correctly

from src.predictor import predict

# Test prediction — will use local joblib since no endpoint is configured
test_input = {
    "type": "M",
    "air_temperature": 298.1,
    "process_temperature": 308.6,
    "rotational_speed": 1551.0,
    "torque": 42.8,
    "tool_wear": 0.0
}

result = predict(test_input)
print(f"Prediction result: {result}")
print(f"Inference backend used: {result['inference_backend']}")
print()

# Test with high-risk inputs
high_risk_input = {
    "type": "L",
    "air_temperature": 304.0,
    "process_temperature": 314.0,
    "rotational_speed": 1200.0,
    "torque": 70.0,
    "tool_wear": 250.0
}

result_high = predict(high_risk_input)
print(f"High risk prediction: {result_high}")
print(f"Inference backend used: {result_high['inference_backend']}")

Prediction result: {'prediction': 0, 'failure_probability': 0.0, 'result': 'NO FAILURE', 'confidence': 1.0, 'inference_backend': 'local'}
Inference backend used: local

High risk prediction: {'prediction': 1, 'failure_probability': 1.0, 'result': 'FAILURE', 'confidence': 1.0, 'inference_backend': 'local'}
Inference backend used: local


In [5]:
# Step 3: Document the SageMaker training workflow
# This cell shows what would happen when SAGEMAKER_ROLE_ARN is set
# and you run: python src/sagemaker_trainer.py

sagemaker_workflow = """
SAGEMAKER TRAINING WORKFLOW
============================

1. Prerequisites:
   - Create S3 bucket: ml-predictive-maintenance-satvik
   - Create SageMaker execution role in IAM with:
       * AmazonSageMakerFullAccess
       * AmazonS3FullAccess
   - Set environment variable:
       export SAGEMAKER_ROLE_ARN=arn:aws:iam::751948408724:role/SageMakerExecutionRole

2. Upload training data to S3:
   - Preprocessed CSV with target column FIRST, no header
   - S3 path: s3://ml-predictive-maintenance-satvik/predictive-maintenance/data/

3. Launch training job (src/sagemaker_trainer.py):
   - Container: AWS built-in XGBoost 1.7-1
   - Instance: ml.m5.large (~$0.115/hour, ~5 min = ~$0.01)
   - Hyperparameters (from local GridSearchCV):
       * num_round: 200
       * max_depth: 5
       * eta: 0.1
       * subsample: 0.8
       * scale_pos_weight: 28
   - Model artifact saved to S3 automatically

4. Deploy endpoint:
   - Instance: ml.m5.large
   - SageMaker manages infrastructure, auto-scaling, health checks
   - Endpoint URL assigned automatically

5. Update FastAPI:
   - Set environment variable: SAGEMAKER_ENDPOINT_NAME=<endpoint-name>
   - src/predictor.py automatically routes to SageMaker
   - No code changes needed

6. IMPORTANT - Delete endpoint after testing:
   - predictor.delete_endpoint(endpoint_name)
   - Running endpoint charges ~$0.115/hour even with no traffic
"""

print(sagemaker_workflow)


SAGEMAKER TRAINING WORKFLOW

1. Prerequisites:
   - Create S3 bucket: ml-predictive-maintenance-satvik
   - Create SageMaker execution role in IAM with:
       * AmazonSageMakerFullAccess
       * AmazonS3FullAccess
   - Set environment variable:
       export SAGEMAKER_ROLE_ARN=arn:aws:iam::751948408724:role/SageMakerExecutionRole

2. Upload training data to S3:
   - Preprocessed CSV with target column FIRST, no header
   - S3 path: s3://ml-predictive-maintenance-satvik/predictive-maintenance/data/

3. Launch training job (src/sagemaker_trainer.py):
   - Container: AWS built-in XGBoost 1.7-1
   - Instance: ml.m5.large (~$0.115/hour, ~5 min = ~$0.01)
   - Hyperparameters (from local GridSearchCV):
       * num_round: 200
       * max_depth: 5
       * eta: 0.1
       * subsample: 0.8
       * scale_pos_weight: 28
   - Model artifact saved to S3 automatically

4. Deploy endpoint:
   - Instance: ml.m5.large
   - SageMaker manages infrastructure, auto-scaling, health checks
   - Endpoint

In [6]:
# Step 4: Verify the sagemaker_trainer.py imports correctly
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "sagemaker_trainer", 
    "src/sagemaker_trainer.py"
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

print("sagemaker_trainer.py imports successfully.")
print(f"Configured region: {module.REGION}")
print(f"Configured bucket: {module.BUCKET_NAME}")
print(f"Configured instance: {module.INSTANCE_TYPE}")
print()
print("To run a real training job:")
print("  1. Set SAGEMAKER_ROLE_ARN environment variable")
print("  2. Create S3 bucket:", module.BUCKET_NAME)
print("  3. Run: python src/sagemaker_trainer.py")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\babaw\\OneDrive\\Desktop\\work\\Project 10 — Classical ML Pipeline\\src/sagemaker_trainer.py'

In [7]:
# Step 4: Verify the sagemaker_trainer.py imports correctly
from src.sagemaker_trainer import REGION, BUCKET_NAME, INSTANCE_TYPE

print("sagemaker_trainer.py imports successfully.")
print(f"Configured region: {REGION}")
print(f"Configured bucket: {BUCKET_NAME}")
print(f"Configured instance: {INSTANCE_TYPE}")
print()
print("To run a real training job:")
print("  1. Set SAGEMAKER_ROLE_ARN environment variable")
print("  2. Create S3 bucket:", BUCKET_NAME)
print("  3. Run: python src/sagemaker_trainer.py")

ModuleNotFoundError: No module named 'src.sagemaker_trainer'

In [9]:
# Step 4: Verify the sagemaker_trainer.py imports correctly
from pathlib import Path

path = Path("src/sagemaker_trainer.py")
print(f"File exists: {path.exists()}")

from src.sagemaker_trainer import REGION, BUCKET_NAME, INSTANCE_TYPE

print("sagemaker_trainer.py imports successfully.")
print(f"Configured region: {REGION}")
print(f"Configured bucket: {BUCKET_NAME}")
print(f"Configured instance: {INSTANCE_TYPE}")

File exists: True
sagemaker_trainer.py imports successfully.
Configured region: us-east-1
Configured bucket: ml-predictive-maintenance-satvik
Configured instance: ml.m5.large
